# CogniVoice Component D — Colab training (PP2: acted + real)

Trains on the clean acted datasets (RAVDESS + CREMA-D + TESS), **downloaded
directly from Kaggle** — no KINGSTON, no Drive upload for the big data — with
the shared preprocessing (loudness norm, 70 Hz high-pass, silence trim)
applied automatically and identically to every clip.

**You only need two things:**
1. Runtime → Change runtime type → **T4 GPU**
2. A Kaggle API token `kaggle.json`: kaggle.com → profile → **Settings → API →
   Create New Token**. Either drop it in Drive at `MyDrive/cognivoice/kaggle.json`,
   or the notebook will ask you to upload it in cell 3.

Run top to bottom. Features + model are cached to Drive so a dropped session
doesn't lose the slow step.

*Optional English augmentation:* drop `real_collected.zip` in `MyDrive/cognivoice/`
(not needed for the first run — acted-only is the key experiment).

In [ ]:
# 1. Mount Drive (used to cache features/model + read kaggle.json)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/cognivoice'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted at', DRIVE)

In [ ]:
# 2. Get the project code + install dependencies (~3 min)
%cd /content
!rm -rf cognivoice-component-d
!git clone -q https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d
!pip install -q funasr modelscope librosa soundfile praat-parselmouth scipy scikit-learn tqdm kaggle
print('setup done')

In [ ]:
# 3. Download acted datasets from Kaggle (RAVDESS + CREMA-D + TESS)
import os
os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists(f'{DRIVE}/kaggle.json'):
    !cp "{DRIVE}/kaggle.json" /root/.kaggle/kaggle.json
else:
    from google.colab import files
    print('Upload kaggle.json (Kaggle > Settings > API > Create New Token):')
    files.upload()
    !cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

os.makedirs('/content/acted', exist_ok=True)
%cd /content/acted
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio && unzip -q -o ravdess-emotional-speech-audio.zip -d RAVDESS
!kaggle datasets download -d ejlok1/cremad && unzip -q -o cremad.zip -d CREMA-D
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess && unzip -q -o toronto-emotional-speech-set-tess.zip -d TESS
%cd /content/cognivoice-component-d
print('RAVDESS + CREMA-D + TESS downloaded to /content/acted')

In [ ]:
# 4. Build metadata (speaker-independent splits). Paths point at the
# downloaded folders; the parsers rglob for wavs so nesting doesn't matter.
import os
os.makedirs('data', exist_ok=True); os.makedirs('models', exist_ok=True)
!python -m src.datasets.acted \
    --ravdess /content/acted/RAVDESS --cremad /content/acted/CREMA-D --tess /content/acted/TESS \
    --out data/metadata_acted.csv

metas = ['data/metadata_acted.csv']
# optional real English augmentation (drop real_collected.zip in Drive)
if os.path.exists(f'{DRIVE}/real_collected.zip'):
    !mkdir -p data/raw/real_collected
    !cd data/raw/real_collected && unzip -q -o "{DRIVE}/real_collected.zip"
    !python -m src.datasets.collected --root data/raw/real_collected --out data/metadata_collected.csv
    if os.path.exists('data/metadata_collected.csv'):
        metas.append('data/metadata_collected.csv')
print('training metadata:', metas)

In [ ]:
# 5. FEATURE EXTRACTION - the slow step (~25-45 min on T4; longer with augment).
# Set AUGMENT>0 to make studio audio resemble PHONE recordings (the acoustic
# domain-gap fix): it adds that many phone-like copies per TRAIN clip, so the
# model learns to read stress through codec/noise/reverb. Re-extracts + caches
# under a separate name so your clean features are kept.
import os
AUGMENT = 0            # try 2 for the domain-gap fix (slower: ~3x train clips)
suffix = f'_aug{AUGMENT}' if AUGMENT else ''
OUT = f'data/features_actedreal{suffix}.npz'
meta_arg = ' '.join(metas)
cached = f"{DRIVE}/{os.path.basename(OUT)}"
if os.path.exists(cached):
    print('features cached in Drive - skipping extraction')
    !cp "{cached}" {OUT}
else:
    !python scripts/extract_features.py --metadata {meta_arg} --out {OUT} --encoder plus_large --augment {AUGMENT}
    !cp {OUT} "{cached}"
    print('features cached to Drive')

In [ ]:
# 6. TRAIN the fusion model — baseline (fast, minutes)
!python scripts/train_fusion.py --features {OUT} --out models/fusion_v2.pt

In [ ]:
# 7. (OPTIONAL) tuned + class-balanced run via Optuna (~20-40 min).
# Acted data is ~balanced, so run only if the baseline needs a lift.
# !pip install -q optuna
# !python scripts/tune_fusion.py --features {OUT} --trials 40 --out models/fusion_tuned.pt

In [ ]:
# 8. Save the trained model + history back to Drive
!cp models/fusion_v2.pt "{DRIVE}/fusion_v2.pt"
!cp models/fusion_v2.history.json "{DRIVE}/fusion_v2.history.json" 2>/dev/null || true
print('saved fusion_v2.pt to Drive — download it into models/ on your Mac')